# KPI Extraction

This notebook extracts financial KPIs from annual reports and stores the results in a CSV file.

## Setup

In [ ]:
%pip install -q pdfplumber pandas regex

## Imports

In [ ]:
import re
from pathlib import Path
from typing import Dict, List

import pandas as pd
import pdfplumber

## Configuration

In [ ]:
ANNUAL_REPORT_DIR = Path('Annual Reports')
OUTPUT_CSV = Path('kpis_extracted.csv')

KPI_PATTERNS = {
    'revenue': r'(revenue|sales|turnover)[^0-9]+([0-9,.]+)',
    'ebit': r'(ebit|operating profit)[^0-9]+([0-9,.]+)',
    'net_income': r'(net income|profit attributable)[^0-9]+([0-9,.]+)',
    'total_assets': r'(total assets)[^0-9]+([0-9,.]+)',
    'scope_1_emissions': r'(scope\s*1[^0-9]*emissions?)[^0-9]+([0-9,.]+)',
    'scope_2_emissions': r'(scope\s*2[^0-9]*emissions?)[^0-9]+([0-9,.]+)',
}

## Helper Functions

In [ ]:
def extract_text_blocks(pdf_path: Path) -> List[str]:
    blocks: List[str] = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=1, y_tolerance=1) or ''
            if text:
                blocks.extend([line.strip() for line in text.split('\n') if line.strip()])
    return blocks


def find_kpis(blocks: List[str], patterns: Dict[str, str]) -> Dict[str, str]:
    values: Dict[str, str] = {}
    joined_text = '\n'.join(blocks)
    for kpi, pattern in patterns.items():
        match = re.search(pattern, joined_text, flags=re.IGNORECASE)
        if match:
            values[kpi] = match.group(2).replace(',', '').strip()
        else:
            values[kpi] = ''
    return values

## Process Reports

In [ ]:
records = []
for pdf_path in sorted(ANNUAL_REPORT_DIR.glob('*.pdf')):
    blocks = extract_text_blocks(pdf_path)
    kpi_values = find_kpis(blocks, KPI_PATTERNS)
    kpi_values['report'] = pdf_path.name
    records.append(kpi_values)

kpis_df = pd.DataFrame(records)
kpis_df.to_csv(OUTPUT_CSV, index=False)
kpis_df.head()

## Save and Inspect

The full set of extracted KPIs is saved in `kpis_extracted.csv`.